# HCM vs Athlete's Heart

Runs the full pipeline in Colab: EchoNet EF/EDV/ESV → synthetic seed cases → CTGAN → labels → model comparison → literature profiles.

**Before you start:** put the EchoNet-Dynamic folder in your Google Drive (request access at https://echonet.github.io/dynamic/). The notebook looks for `MyDrive/EchoNet-Dynamic/FileList.csv`.

> Research and teaching code trained on synthetic data. Not for clinical use.

## 1. Install

In [ ]:
!git clone -q https://github.com/nav92061/Science-Fair.git
%cd Science-Fair
!pip install -q ".[ctgan]"

## 2. Find FileList.csv in Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from hcm_athlete.data import find_filelist
FILELIST = find_filelist()
print('Using:', FILELIST)
assert FILELIST, 'FileList.csv not found. Put EchoNet-Dynamic in MyDrive or set FILELIST to its path.'

## 3. Run

CTGAN with 300 epochs takes a few minutes on CPU. Set `synth='bootstrap'` for a 30-second test run (do not report those results).

In [ ]:
from hcm_athlete.pipeline import RunConfig, run

cfg = RunConfig(echonet=str(FILELIST), synth='ctgan', ctgan_epochs=300, seed=42, out_dir='results')
summary = run(cfg)

## 4. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv('results/model_comparison.csv').round(3))
display(pd.read_csv('results/literature_profiles.csv'))
display(Image('results/roc_curves.png', width=600))
display(Image('results/feature_importance.png', width=600))

## 5. Score one case

Any subset of the 45 inputs works. Missing ones are set to the training median.

In [ ]:
from hcm_athlete.model import HCMModel
from hcm_athlete.report import make_report

model = HCMModel.load('results/model.joblib')
case = {'wall_thickness': 14.0, 'lv_cavity': 45.0, 'gls': -12.0, 'e_e_prime': 11.0,
        'EF': 65, 'EDV': 70, 'ESV': 28, 'resting_hr': 70}
p, filled = model.predict_one(case)
print(make_report(case, p, filled, model.name))

## 6. Download results

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')